## Setup start 

In [ ]:
source(file.path("~/workspace/pipelines/snt_dhis2_formatting/utils/snt_dhis2_formatting.r"))
setup_var <- get_setup_variables(packages=c("lubridate", "zoo", "glue", "arrow", "dplyr", "tidyr", "stringr", "stringi", "jsonlite", "reticulate"))
config_json <- load_snt_config(file.path(setup_var$CONFIG_PATH, "SNT_config.json"))

# Save this country code in a variable
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ADMIN_1 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_1)
ADMIN_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)

### Load DHIS2 analytics data  

-Load DHIS2 anlytics from latest dataset version 

In [ ]:
# DHIS2 Dataset extract identifier
dataset_name <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_EXTRACTS
dhis2_data <- load_dataset_file(dataset_name, paste0(COUNTRY_CODE, "_dhis2_raw_analytics.parquet"))

## SNT Indicators computation

### Select dhis2 metadata  

In [ ]:
# log
log_msg("Computing SNT indicators.")

In [ ]:
# Select only metadata (reduce the size of the dataframe)
administrative_cols <- colnames(dhis2_data)[grepl("LEVEL_", colnames(dhis2_data))]
dhis2_metadata <- dhis2_data[ , c("OU", "DX_NAME", administrative_cols)] # Metadata
dhis2_metadata <- distinct(dhis2_metadata)

# Max admin columns available (matchin ou)
name_cols <- grep("LEVEL_\\d+_NAME", administrative_cols, value = TRUE)
max_level <- max(as.numeric(gsub("LEVEL_(\\d+)_NAME", "\\1", name_cols)))
max_admin_col_name <- paste0("LEVEL_", max_level, "_NAME")

# Result
print(glue("Max admin level available: '{max_admin_col_name}'"))
print(glue("Data element names: \n {paste(unique(dhis2_metadata$DX_NAME), collapse=',\n ')}"))

# Clean strings for admin 1 and admin 2 (format_names() in snt_utils.r)
dhis2_metadata[[ADMIN_1]] <- format_names(dhis2_metadata[[ADMIN_1]]) 
dhis2_metadata[[ADMIN_2]] <- format_names(dhis2_metadata[[ADMIN_2]])

### Select dhis2 values data  

In [ ]:
# dhis2 Values table
dhis2_values <- dhis2_data[ , c("DX", "CO", "OU", "PE", "VALUE")]
head(dhis2_values)

### Pivot dhis2 value table

In [ ]:
# make sure we have numeric data in "values" column
dhis2_values$VALUE <- as.numeric(dhis2_values$VALUE)

# pivot table on DX and CO columns (available combinations to columns)
routine_data <- pivot_wider(dhis2_values,
                            id_cols = all_of(c("OU", "PE")),
                            names_from = c("DX", "CO"),
                            values_from = 'VALUE')

print(paste("Routine data pivot : ", paste0(dim(routine_data), collapse=", ")))

### Build indicator definitions

In [ ]:
# copy
routine_data_ind <- routine_data

# Get list of indicator definitions from SNT configuration
dhis_indicator_definitions <- config_json$DHIS2_DATA$DHIS2_INDICATOR_DEFINITIONS
names(dhis_indicator_definitions) <- toupper(names(dhis_indicator_definitions))

**Remove empty indicators from the list**

In [ ]:
dhis_indicator_definitions_clean <- dhis_indicator_definitions
empty_indicators <- c()

# Loop over the indicators and clean the list
for (name in names(dhis_indicator_definitions_clean)) {
  value <- dhis_indicator_definitions_clean[[name]]
  
  # If value is NULL or length zero, leave as is or set to NULL
  if (is.null(value) || length(value) == 0 || all(value == "")) {
    dhis_indicator_definitions_clean[[name]] <- NULL 
    empty_indicators <- c(empty_indicators, name)
    next
  }  
  # Trim whitespace and then check if empty string
  value_trimmed <- trimws(value)
  dhis_indicator_definitions_clean[[name]] <- value_trimmed  
}

print("Complete indicator definitions:")
print(names(dhis_indicator_definitions_clean))
print("Empty indicator definitions: ")
print(empty_indicators)

**Start indicators loop**

In [ ]:
# loop over the definitions
empty_data_indicators <- c()
for (indicator in names(dhis_indicator_definitions_clean)) {
        
    data_element_uids <- dhis_indicator_definitions_clean[[indicator]]    
    col_names <- c()

    if (length(data_element_uids) > 0) {
        for (dx in data_element_uids) {
            dx_co <- gsub("\\.", "_", dx)            
            if (grepl("_", dx_co)) {
                col_names <- c(col_names , dx_co)
            } else {
                if (!any(grepl(dx, colnames(routine_data_ind)))) {  # is there no dx what match?
                    msg <- paste0("Data element : " , dx, " of indicator ", indicator , " is missing in the DHIS2 routine data.")
                    log_msg(msg, level="warning")
                } else {
                    col_names <- c(col_names , colnames(routine_data_ind)[grepl(dx, colnames(routine_data_ind))])
                }                
            }
        }
    
        # check if there are matching data elements
        if (length(col_names) == 0) {
            msg <- paste0("No data elements available to build indicator : " , indicator, ", skipped.")
            log_msg(msg, level="warning")
            empty_data_indicators <- c(empty_data_indicators, indicator)
            next
        }
        
        # logs
        msg <- paste0("Building indicator : ", indicator, " -> column selection : ", paste(col_names, collapse = ", "))        
        log_msg(msg)
        
        if (length(col_names) > 1) {
            sums <- rowSums(routine_data_ind[, col_names], na.rm = TRUE)
            all_na <- rowSums(!is.na(routine_data_ind[, col_names])) == 0
            sums[all_na] <- NA  # Keep NA if all rows are NA!
            routine_data_ind[[indicator]] <- sums            
        } else {
            routine_data_ind[indicator] <- routine_data_ind[, col_names] 
        }
        
    } else {
        routine_data_ind[indicator] <- NA
        
        # logs
        msg <- paste0("Building indicator : ", indicator, " -> column selection : NULL")
        log_msg(msg)
    }
}

In [ ]:
# Add the empty indicator columns (if not needed this can be commented)
for (empty_indicator in empty_indicators) {
    routine_data_ind[empty_indicator] <- NA
    
    # logs
    msg <- paste0("Building indicator : ", empty_indicator, " -> column selection : NULL")
    log_msg(msg)
}

In [ ]:
print(dim(routine_data_ind))
head(routine_data_ind, 3)

## Format SNT routine data

### SNT format 

In [ ]:
# Filter routine data columns by indicators
built_indicators <- names(dhis_indicator_definitions)[!(names(dhis_indicator_definitions) %in% empty_data_indicators)]
routine_data_selection <- routine_data_ind[, c("OU", "PE", built_indicators)]

In [ ]:
# left join with metadata
routine_data_merged <- merge(routine_data_selection, dhis2_metadata, by = "OU", all.x = TRUE)

# Select administrative columns
adm_1_id_col <- gsub("_NAME", "_ID", ADMIN_1)
adm_1_name_col <- ADMIN_1
adm_2_id_col <- gsub("_NAME", "_ID", ADMIN_2)
adm_2_name_col <- ADMIN_2

# Select and Rename
routine_data_formatted <- routine_data_merged %>%
    mutate(        
        YEAR = as.numeric(substr(PE, 1, 4)),
        MONTH = as.numeric(substr(PE, 5, 6)),
        PE = as.numeric(PE)
    ) %>%
    select(
        PERIOD = PE,
        YEAR,
        MONTH,
        OU_ID = OU,
        OU_NAME = !!sym(max_admin_col_name),
        ADM1_NAME = !!sym(adm_1_name_col),
        ADM1_ID = !!sym(adm_1_id_col),
        ADM2_NAME = !!sym(adm_2_name_col),
        ADM2_ID = !!sym(adm_2_id_col),
        all_of(built_indicators)
    )

# Column names to upper case
colnames(routine_data_formatted) <- clean_column_names(routine_data_formatted)

# Sort dataframe by period
routine_data_formatted <- routine_data_formatted[order(as.numeric(routine_data_formatted$PERIOD)), ]
print(dim(routine_data_formatted))

In [ ]:
# Add the empty data indicator columns (if not needed this can be commented)
for (empty_data_indicator in empty_data_indicators) {
    routine_data_formatted[empty_data_indicator] <- NA    
    # logs
    print(paste0("Set indicator ", empty_data_indicator, " to : NULL"))
    # log_msg(msg)
}

In [ ]:
head(routine_data_formatted,3)

### Output formatted routine data

In [ ]:
out_msg <- paste0("Rountine data saved under: ", file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_routine.parquet")))

# write parquet file
write_parquet(routine_data_formatted, file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_routine.parquet")))

# write csv file
write.csv(routine_data_formatted, file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_routine.csv")), row.names = FALSE)

In [ ]:
# log
log_msg(out_msg)

### Data Summary 

In [ ]:
# Data summary
print(summary(routine_data_formatted))